# Oracle Connection Selection

Establishes the **theoretical upper bound** of reconstruction quality by using the GT topology as an oracle to score candidate inter-fragment connections.

**Pipeline:**
1. Extract seeds from annotation using `TopologyBuilder`
2. Build all candidate inter-fragment connections within `r = 50 px` using `PathFinder`
3. Score each connection: `score = mean distance of path points to nearest GT point`
4. Sweep score threshold → select connections below threshold → compute Hausdorff vs GT
5. Compare: *baseline* (seeds only) vs *oracle* (optimal threshold) vs *all edges*

> **Point set for Hausdorff**: Only **edge path points** are used — isolated seed nodes
> (unconnected fragments) are excluded. The metric reflects purely whether the selected
> connections follow the correct paths, not where isolated seeds happen to land.

In [ ]:
import warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from scipy.spatial import KDTree
from pathlib import Path
from skimage.measure import label as sk_label

from neural_reconstruction.core.topology.topology_builder import TopologyBuilder
from neural_reconstruction.core.pathfinding.path_finder import PathFinder
from neural_reconstruction.core.evaluation import (
    GraphPointExtractor,
    extract_graph_points,
    compute_average_hausdorff_distance,
    compute_point_min_distances,
)

warnings.filterwarnings('ignore')


def extract_edge_points(graph):
    """Extract only edge path points (no isolated nodes) for Hausdorff evaluation.

    Isolated seed nodes that have no connections do not contribute to the metric,
    so only the geometry of the selected edges is evaluated.
    Returns (0, 2) array if the graph has no edges.
    """
    extractor = GraphPointExtractor(remove_duplicates=True)
    return extractor.extract_points(graph, include_nodes=False, include_edge_paths=True)

In [ ]:
IMAGE_ID       = 'S1585-2_a'
BASE_PATH      = Path(f'../data/{IMAGE_ID}')
SEARCH_RADIUS  = 100
SEGMENT_LENGTH = 3.0

## 1. Load Input Data

In [ ]:
image_rgb  = cv2.imread(str(BASE_PATH / 'image.png'),      cv2.IMREAD_COLOR_RGB)
mask       = cv2.imread(str(BASE_PATH / 'mask.png'),       cv2.IMREAD_GRAYSCALE)
annotation = cv2.imread(str(BASE_PATH / 'annotation.png'), cv2.IMREAD_GRAYSCALE)
label_img  = cv2.imread(str(BASE_PATH / 'new_label.png'),      cv2.IMREAD_GRAYSCALE)

green = image_rgb[:, :, 1]  # green channel (strongest nerve fiber signal)

print(f'Image      shape: {image_rgb.shape},  dtype: {image_rgb.dtype}')
print(f'Mask       shape: {mask.shape},       dtype: {mask.dtype}')
print(f'Annotation shape: {annotation.shape}, dtype: {annotation.dtype}')
print(f'Label      shape: {label_img.shape},  dtype: {label_img.dtype}')

fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
axes[0].imshow(green,      cmap='gray');  axes[0].set_title('Green Channel'); axes[0].axis('off')
axes[1].imshow(mask,       cmap='gray');  axes[1].set_title('Mask');          axes[1].axis('off')
axes[2].imshow(annotation, cmap='gray');  axes[2].set_title('Annotation');    axes[2].axis('off')
axes[3].imshow(label_img,  cmap='gray');  axes[3].set_title('GT Label');      axes[3].axis('off')
plt.show()

## 2. Seed Extraction

Use `TopologyBuilder` to skeletonize the annotation and subdivide skeleton edges into seeds of spacing `segment_length`.

In [ ]:
builder    = TopologyBuilder(segment_length=SEGMENT_LENGTH)
seed_graph = builder.build_seed_graph(annotation)

seed_nodes  = list(seed_graph.nodes())       # list of (y, x) tuples
seed_coords = np.array(seed_nodes, dtype=int)  # (N, 2)

num_components = nx.number_connected_components(seed_graph)

print(f'Seeds (nodes):         {seed_graph.number_of_nodes()}')
print(f'Intra-fragment edges:  {seed_graph.number_of_edges()}')
print(f'Connected components:  {num_components}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8), constrained_layout=True)

# Left: annotation
axes[0].imshow(annotation, cmap='gray')
axes[0].set_title('Annotation (binary)')
axes[0].axis('off')

# Right: seeds overlaid on green channel
axes[1].imshow(green, cmap='gray')
for u, v, data in seed_graph.edges(data=True):
    path = data.get('path', [u, v])
    if len(path) >= 2:
        axes[1].plot([p[1] for p in path], [p[0] for p in path],
                     color='lime', linewidth=0.8, alpha=0.7)
axes[1].scatter(seed_coords[:, 1], seed_coords[:, 0],
                c='cyan', s=2, alpha=0.6, zorder=3)
axes[1].set_title(f'Seeds on Green Channel  ({len(seed_nodes)} seeds, {num_components} components)')
axes[1].axis('off')

plt.show()

## 3. Build Candidate Inter-Fragment Connections

Use `PathFinder` with `search_radius = 50 px` to find all A*-paths between seeds belonging to **different** annotation fragments.

- Cost map: `(255 - green) / 255`  (lower cost = brighter = easier to traverse)
- `label_img` ensures same-fragment seed pairs are skipped

In [ ]:
cost_map    = ((255.0 - green.astype(np.float64)) / 255.0) ** 2
comp_labels = sk_label((annotation > 0).astype(np.uint8), connectivity=2)

kdtree = KDTree(seed_coords)

finder      = PathFinder(cost_map)
path_lookup = finder.find_paths_from_seeds(
    topology_points=seed_coords,
    kdtree=kdtree,
    search_radius=SEARCH_RADIUS,
    label_img=comp_labels,
)

print(f'Candidate inter-fragment connections: {len(path_lookup)}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10), constrained_layout=True)
ax.imshow(green, cmap='gray')

for (u, v), (path, cost) in path_lookup.items():
    if len(path) >= 2:
        ax.plot([p[1] for p in path], [p[0] for p in path],
                color='yellow', linewidth=0.5, alpha=0.4)

# Draw intra-fragment skeleton on top
for u, v, data in seed_graph.edges(data=True):
    path = data.get('path', [u, v])
    if len(path) >= 2:
        ax.plot([p[1] for p in path], [p[0] for p in path],
                color='lime', linewidth=1.0, alpha=0.8)

ax.set_title(f'Candidate Connections (yellow, n={len(path_lookup)})  +  Annotation Seeds (green)')
ax.axis('off')
plt.show()

## 4. Extract GT Topology

Build the GT topology from `label.png` using the same `TopologyBuilder` and extract its
**edge path point set** (nodes excluded, consistent with prediction evaluation).

In [ ]:
gt_builder = TopologyBuilder(segment_length=SEGMENT_LENGTH)
gt_graph   = gt_builder.build_seed_graph(label_img)
gt_points  = extract_edge_points(gt_graph)   # edge path points only (M, 2)
gt_kdtree  = KDTree(gt_points)

print(f'GT Nodes:        {gt_graph.number_of_nodes()}')
print(f'GT Edges:        {gt_graph.number_of_edges()}')
print(f'GT Edge Points:  {len(gt_points)} (edge path points only, nodes excluded)')

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(20, 8), constrained_layout=True)

axes[0].imshow(label_img, cmap='gray')
axes[0].set_title('GT Label Image')
axes[0].axis('off')

axes[1].imshow(green, cmap='gray')
for u, v, data in gt_graph.edges(data=True):
    path = data.get('path', [u, v])
    if len(path) >= 2:
        axes[1].plot([p[1] for p in path], [p[0] for p in path],
                     color='tomato', linewidth=0.8, alpha=0.8)
gt_nodes_arr = np.array(list(gt_graph.nodes()))
axes[1].scatter(gt_nodes_arr[:, 1], gt_nodes_arr[:, 0],
                c='orange', s=2, alpha=0.6, zorder=3)
axes[1].set_title(f'GT Topology  ({gt_graph.number_of_nodes()} nodes, {gt_graph.number_of_edges()} edges)')
axes[1].axis('off')

plt.show()

## 5. Two-Directional Edge Scoring

Pre-compute the two quantities used by the greedy selection in section 6:

**Pred→GT cost** — how much do path points wander away from GT?
$$\text{pred\_cost}(e) = \sum_{p \in P_e} \min_{q \in \text{GT}} \|p - q\|$$

**GT→pred coverage gain** — how much uncovered GT area does this edge recover? ($d_0(q)$ = initial GT→pred from seed edges only)
$$\text{coverage\_gain}(e) = \sum_{q \in \text{GT}} \max\!\left(0,\; d_0(q) - \min_{p \in P_e}\|q - p\|\right)$$

These are also combined into a **coverage efficiency ratio** $= \text{coverage\_gain} / \text{pred\_cost}$ for inspection.

In [ ]:
from scipy.spatial.distance import cdist as _cdist

# --- Initial GT→pred distances from seed_graph intra-fragment edges only ---
seed_edge_pts = extract_edge_points(seed_graph)
if len(seed_edge_pts) > 0:
    initial_gt_to_pred = KDTree(seed_edge_pts).query(gt_points)[0]  # (M,)
else:
    initial_gt_to_pred = np.full(len(gt_points), np.inf)

print(f'Initial GT→Pred (seed edges only):  mean={initial_gt_to_pred.mean():.2f} px, '
      f'max={initial_gt_to_pred.max():.2f} px')

# --- Score each candidate edge on both directions ---
edge_pred_cost    = {}   # sum of path-point distances to GT
edge_coverage_gain = {}  # sum of GT→pred improvements

for (u, v), (path, _cost) in path_lookup.items():
    path_arr = np.array(path, dtype=np.float64)   # (K, 2)

    # pred→GT cost: sum of min-distances from path points to GT
    d_p2g     = _cdist(path_arr, gt_points).min(axis=1)   # (K,)
    pred_cost = float(d_p2g.sum())

    # GT→pred coverage gain: GT points pulled closer by this edge
    d_g2path = _cdist(gt_points, path_arr).min(axis=1)    # (M,)
    gain     = np.maximum(0.0, initial_gt_to_pred - d_g2path)
    cov_gain = float(gain.sum())

    edge_pred_cost[(u, v)]     = pred_cost
    edge_coverage_gain[(u, v)] = cov_gain

edge_keys  = list(path_lookup.keys())
pred_costs = np.array([edge_pred_cost[k]     for k in edge_keys], dtype=float)
cov_gains  = np.array([edge_coverage_gain[k] for k in edge_keys], dtype=float)

# Coverage efficiency ratio: gain / cost  (inf when pred_cost=0 → perfectly aligned with GT)
with np.errstate(divide='ignore', invalid='ignore'):
    efficiency = np.where(pred_costs > 0, cov_gains / pred_costs, np.inf)

efficiency_dict = dict(zip(edge_keys, efficiency))

finite_eff = efficiency[np.isfinite(efficiency)]
print(f'\nPred Cost       median={np.median(pred_costs):.1f}, mean={pred_costs.mean():.1f}')
print(f'Coverage Gain   median={np.median(cov_gains):.1f}, mean={cov_gains.mean():.1f}')
print(f'Efficiency      median={np.median(finite_eff):.2f}, mean={finite_eff.mean():.2f}')
print(f'Edges where gain > cost (eff > 1):  {(efficiency > 1).sum()} / {len(edge_keys)}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5), constrained_layout=True)

# Left: Pred Cost distribution
axes[0].hist(pred_costs, bins=50, color='tomato', edgecolor='white', linewidth=0.4)
axes[0].axvline(np.median(pred_costs), color='black', linestyle='--',
                label=f'Median = {np.median(pred_costs):.0f}')
axes[0].set_xlabel('Pred Cost (sum of path→GT distances, px)')
axes[0].set_ylabel('Number of Edges')
axes[0].set_title('Pred→GT Cost Distribution\n(lower = edge closely follows GT)')
axes[0].legend()

# Centre: Coverage Gain distribution
axes[1].hist(cov_gains, bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].axvline(np.median(cov_gains), color='black', linestyle='--',
                label=f'Median = {np.median(cov_gains):.0f}')
axes[1].set_xlabel('Coverage Gain (sum of GT→pred improvements, px)')
axes[1].set_ylabel('Number of Edges')
axes[1].set_title('GT→Pred Coverage Gain Distribution\n(higher = covers more uncovered GT area)')
axes[1].legend()

# Right: Efficiency ratio (clipped)
axes[2].hist(np.clip(finite_eff, 0, 10), bins=50, color='mediumpurple',
             edgecolor='white', linewidth=0.4)
axes[2].axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='λ = 1 (break-even)')
axes[2].axvline(np.median(finite_eff), color='black', linestyle='--',
                label=f'Median = {np.median(finite_eff):.2f}')
axes[2].set_xlabel('Coverage Efficiency (gain / cost), clipped at 10')
axes[2].set_ylabel('Number of Edges')
axes[2].set_title('Coverage Efficiency Distribution\n(> 1 = edge reduces total symmetric Hausdorff)')
axes[2].legend()

plt.show()

In [ ]:
# Side-by-side: Pred Cost (left) vs Coverage Gain (right)
pred_norm = mcolors.Normalize(vmin=pred_costs.min(), vmax=np.percentile(pred_costs, 90))
cov_norm  = mcolors.Normalize(vmin=0, vmax=np.percentile(cov_gains, 90))

fig, axes = plt.subplots(1, 2, figsize=(28, 10), constrained_layout=True)

for ax, edge_val_dict, norm_obj, cmap_obj, title, cbar_label in [
    (axes[0], edge_pred_cost,    pred_norm, plt.cm.RdYlGn_r,
     'Pred→GT Cost per Edge\n(green = follows GT, red = wanders away)',
     'Pred Cost (sum of path→GT distances)'),
    (axes[1], edge_coverage_gain, cov_norm, plt.cm.YlOrRd,
     'GT→Pred Coverage Gain per Edge\n(dark = covers many uncovered GT points)',
     'Coverage Gain (sum of GT→pred improvements)'),
]:
    ax.imshow(green, cmap='gray')
    for (u, v), (path, _cost) in path_lookup.items():
        if len(path) >= 2:
            color = cmap_obj(norm_obj(edge_val_dict[(u, v)]))
            ax.plot([p[1] for p in path], [p[0] for p in path],
                    color=color, linewidth=0.8, alpha=0.6)
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm_obj)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label=cbar_label, fraction=0.02, pad=0.01)
    ax.set_title(title)
    ax.axis('off')

plt.show()

## 6. Greedy Edge Selection (Direct Hausdorff Minimisation)

At each step, select the candidate edge that **directly minimises the Average Hausdorff Distance** when added to the current graph. Stop when no edge can reduce it further.

**State tracked across steps**

| Variable | Meaning |
|----------|---------|
| `current_gt_dists[q]` | Current distance from GT point $q$ to nearest pred point |
| `current_p2g_total` | Sum of pred→GT min-distances across all current pred points |
| `current_n_pred` | Total number of pred points currently in graph |

**Selection criterion per step**

$$d_{\text{pred→GT}}^{\text{new}}(e) = \frac{\text{current\_p2g\_total} + \text{pred\_cost}(e)}{N_{\text{pred}} + |\text{path}(e)|}$$

$$d_{\text{GT→pred}}^{\text{new}}(e) = \frac{1}{M}\sum_q \min\!\bigl(d_{\text{current}}(q),\; g2p[e,q]\bigr)$$

$$e^* = \arg\min_e \;\frac{d_{\text{pred→GT}}^{\text{new}}(e) + d_{\text{GT→pred}}^{\text{new}}(e)}{2}$$

Stop when $H^{\text{new}}(e^*) \geq H^{\text{current}}$.

In [ ]:
# ── Baselines ──────────────────────────────────────────────────────────────
baseline_pts  = extract_edge_points(seed_graph)
baseline_dist = (compute_average_hausdorff_distance(baseline_pts, gt_points)
                 if len(baseline_pts) else np.inf)

all_edges_graph = seed_graph.copy()
for (u, v), (path, cost) in path_lookup.items():
    all_edges_graph.add_edge(u, v, path=path, weight=cost)
all_edges_pts  = extract_edge_points(all_edges_graph)
all_edges_dist = (compute_average_hausdorff_distance(all_edges_pts, gt_points)
                  if len(all_edges_pts) else np.inf)

print(f'Baseline (intra-fragment only): {baseline_dist:.4f} px  ({len(baseline_pts)} pts)')
print(f'All edges (no filter):          {all_edges_dist:.4f} px  ({len(all_edges_pts)} pts)')

# ── Pre-filter candidates ──────────────────────────────────────────────────
# Only keep edges that could possibly improve GT→pred coverage
N_MAX = 8000   # memory cap: N_MAX × M × 4 bytes (float32)

candidate_keys = [k for k in edge_keys if edge_coverage_gain[k] > 0]
print(f'\nEdges with coverage_gain > 0: {len(candidate_keys)} / {len(edge_keys)}')

if len(candidate_keys) > N_MAX:
    with np.errstate(divide='ignore', invalid='ignore'):
        efficiency_dict = {k: (edge_coverage_gain[k] / edge_pred_cost[k]
                               if edge_pred_cost[k] > 0 else np.inf)
                           for k in edge_keys}
    eff_vals = [efficiency_dict[k] if np.isfinite(efficiency_dict[k]) else 1e9
                for k in candidate_keys]
    top_idx = np.argsort(eff_vals)[::-1][:N_MAX]
    candidate_keys = [candidate_keys[i] for i in top_idx]
    print(f'Capped to top-{N_MAX} by efficiency ratio.')

N_cand = len(candidate_keys)

# ── Precompute per-candidate arrays ───────────────────────────────────────
candidate_pred_costs_f32 = np.array(
    [edge_pred_cost[k] for k in candidate_keys], dtype=np.float32)
candidate_path_lens = np.array(
    [len(path_lookup[k][0]) for k in candidate_keys], dtype=np.float32)

# g2p_mat[i, q] = distance from GT point q to nearest point on edge i
# Shape: (N_cand, M)  float32
gt_pts_f32 = gt_points.astype(np.float32)
g2p_mat = np.empty((N_cand, len(gt_points)), dtype=np.float32)

print(f'\nPrecomputing g2p_mat ({N_cand} × {len(gt_points)}, '
      f'{N_cand * len(gt_points) * 4 / 1e6:.0f} MB)...')
for i, (u, v) in enumerate(candidate_keys):
    path_arr = np.array(path_lookup[(u, v)][0], dtype=np.float32)
    g2p_mat[i] = _cdist(gt_pts_f32, path_arr).min(axis=1)
    if (i + 1) % 1000 == 0:
        print(f'  {i + 1}/{N_cand}', end='\r')
print(f'  Done.')

# ── Initialise tracking state ──────────────────────────────────────────────
current_gt_dists = initial_gt_to_pred.astype(np.float32).copy()   # (M,)

# Current pred→GT: sum of min-distances from each seed-edge point to GT
if len(baseline_pts) > 0:
    seed_p2g_dists    = _cdist(baseline_pts.astype(np.float32), gt_pts_f32).min(axis=1)
    current_p2g_total = float(seed_p2g_dists.sum())
    current_n_pred    = len(baseline_pts)
else:
    current_p2g_total = 0.0
    current_n_pred    = 0

current_d_p2g = current_p2g_total / max(1, current_n_pred)
current_d_g2p = float(current_gt_dists.mean())
current_H     = (current_d_p2g + current_d_g2p) / 2.0
print(f'\nInitial  H={current_H:.4f}  '
      f'(pred→GT={current_d_p2g:.4f}, GT→pred={current_d_g2p:.4f})')

# ── Greedy selection loop ──────────────────────────────────────────────────
remaining_mask   = np.ones(N_cand, dtype=bool)
selected_indices = []
hausdorff_history = [current_H]   # H value after each selection
gain_history      = []             # ΔH improvement per step (positive = improvement)

print('Running greedy...')
while True:
    # New GT→pred for every candidate (vectorised)
    new_d_g2p = np.minimum(current_gt_dists, g2p_mat).mean(axis=1)     # (N_cand,)

    # New pred→GT for every candidate
    new_p2g_total = current_p2g_total + candidate_pred_costs_f32        # (N_cand,)
    new_n_pred    = current_n_pred    + candidate_path_lens              # (N_cand,)
    new_d_p2g     = new_p2g_total / new_n_pred                          # (N_cand,)

    # New symmetric Hausdorff
    new_H = (new_d_p2g + new_d_g2p) / 2.0                              # (N_cand,)
    new_H[~remaining_mask] = np.inf   # exclude already-selected

    best     = int(np.argmin(new_H))
    best_H   = float(new_H[best])

    if best_H >= current_H:   # no improvement possible
        break

    delta_H = current_H - best_H   # positive = improvement
    selected_indices.append(best)
    remaining_mask[best] = False

    # Update state
    current_gt_dists   = np.minimum(current_gt_dists, g2p_mat[best])
    current_p2g_total += float(candidate_pred_costs_f32[best])
    current_n_pred    += int(candidate_path_lens[best])
    current_H          = best_H

    hausdorff_history.append(current_H)
    gain_history.append(delta_H)

    if len(selected_indices) % 200 == 0:
        print(f'  step {len(selected_indices):5d}  H={current_H:.4f}  ΔH={delta_H:.4f}', end='\r')

selected_greedy_keys = [candidate_keys[i] for i in selected_indices]
print(f'\nGreedy selected {len(selected_greedy_keys)} edges '
      f'(from {N_cand} candidates).')
print(f'Final    H={current_H:.4f}  '
      f'(improved by {baseline_dist - current_H:.4f} px from baseline)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

steps = np.arange(len(hausdorff_history))   # 0 = initial (before any selection)

# Left: Hausdorff convergence curve
ax = axes[0]
ax.plot(steps, hausdorff_history, color='steelblue', linewidth=2.0, label='Oracle (Greedy)')
ax.axhline(baseline_dist,  color='gray',   linestyle='--', linewidth=1.5,
           label=f'Baseline = {baseline_dist:.4f} px')
ax.axhline(all_edges_dist, color='salmon', linestyle='--', linewidth=1.5,
           label=f'All edges = {all_edges_dist:.4f} px')
ax.axhline(hausdorff_history[-1], color='limegreen', linestyle=':',  linewidth=1.5,
           label=f'Final H = {hausdorff_history[-1]:.4f} px')
ax.set_xlabel('Number of Selected Edges')
ax.set_ylabel('Average Hausdorff Distance (px)')
ax.set_title('Hausdorff Convergence\n(direct greedy minimisation)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: ΔH (improvement) per step
ax2 = axes[1]
if gain_history:
    ax2.plot(np.arange(1, len(gain_history) + 1), gain_history,
             color='darkorange', linewidth=1.2, alpha=0.8)
ax2.axhline(0, color='gray', linestyle='--', linewidth=1.0)
ax2.set_xlabel('Greedy Step')
ax2.set_ylabel('ΔH per Step (px improvement)')
ax2.set_title('Marginal Hausdorff Improvement per Step\n(algorithm stops when this reaches 0)')
ax2.grid(True, alpha=0.3)

fig.suptitle(f'Greedy Hausdorff Minimisation — {IMAGE_ID}', fontsize=13)
plt.show()

print(f'Greedy selected: {len(selected_greedy_keys)} edges')
print(f'Hausdorff:  baseline={baseline_dist:.4f}  →  oracle={hausdorff_history[-1]:.4f} px'
      f'  (Δ = {baseline_dist - hausdorff_history[-1]:.4f} px)')

## 7. Final Evaluation

Compare three configurations at the optimal threshold:

In [ ]:
# Build oracle graph from greedy-selected edges
oracle_graph = seed_graph.copy()
for (u, v) in selected_greedy_keys:
    path, cost = path_lookup[(u, v)]
    oracle_graph.add_edge(u, v, path=path, weight=cost)

configs = [
    ('Baseline (intra only)',  seed_graph,       baseline_pts),
    ('Oracle (Greedy)',        oracle_graph,      None),
    ('All Edges (r=50)',       all_edges_graph,   all_edges_pts),
]

print(f'{"Config":<28}  {"Nodes":>6}  {"Edges":>6}  {"EdgePts":>8}  '
      f'{"Avg Hausdorff":>14}  {"Pred→GT":>9}  {"GT→Pred":>9}')
print('-' * 98)

for name, graph, pts in configs:
    if pts is None:
        pts = extract_edge_points(graph)
    if len(pts) == 0:
        print(f'{name:<28}  {graph.number_of_nodes():>6}  {graph.number_of_edges():>6}  '
              f'{len(pts):>8}  {"(no edges)":>14}')
        continue
    avg, d_p2g, d_g2p = compute_average_hausdorff_distance(pts, gt_points, return_components=True)
    print(f'{name:<28}  {graph.number_of_nodes():>6}  {graph.number_of_edges():>6}  '
          f'{len(pts):>8}  {avg:>14.4f}  {d_p2g:>9.4f}  {d_g2p:>9.4f}')

print('-' * 98)
print(f'{"GT":<28}  {gt_graph.number_of_nodes():>6}  {gt_graph.number_of_edges():>6}  '
      f'{len(gt_points):>8}  {"(reference)":>14}')

## 8. Comparison Visualization

In [ ]:
def draw_graph_on_ax(ax, background, graph, edge_color='lime', node_color='cyan',
                     linewidth=1.0, node_size=1.5, title=''):
    ax.imshow(background, cmap='gray')
    for u, v, data in graph.edges(data=True):
        path = data.get('path', [u, v])
        if len(path) >= 2:
            ax.plot([p[1] for p in path], [p[0] for p in path],
                    color=edge_color, linewidth=linewidth, alpha=0.8)
    nodes_arr = np.array(list(graph.nodes()))
    if len(nodes_arr) > 0:
        ax.scatter(nodes_arr[:, 1], nodes_arr[:, 0],
                   c=node_color, s=node_size, alpha=0.6, zorder=3)
    ax.set_title(title)
    ax.axis('off')


# Compute oracle Hausdorff for title
oracle_edge_pts = extract_edge_points(oracle_graph)
if len(oracle_edge_pts) > 0:
    oracle_avg, oracle_p2g, oracle_g2p = compute_average_hausdorff_distance(
        oracle_edge_pts, gt_points, return_components=True)
else:
    oracle_avg = oracle_p2g = oracle_g2p = np.inf

fig, axes = plt.subplots(2, 2, figsize=(24, 16), constrained_layout=True)

draw_graph_on_ax(
    axes[0, 0], green, gt_graph,
    edge_color='tomato', node_color='orange', linewidth=1.2, node_size=2,
    title=f'GT Topology  ({gt_graph.number_of_nodes()} nodes, {gt_graph.number_of_edges()} edges)'
)
draw_graph_on_ax(
    axes[0, 1], green, seed_graph,
    edge_color='lime', node_color='cyan', linewidth=1.0, node_size=1.5,
    title=f'Baseline — Intra-Fragment Only\n'
          f'Hausdorff={baseline_dist:.2f} px  ({len(baseline_pts)} edge pts)'
)
draw_graph_on_ax(
    axes[1, 0], green, oracle_graph,
    edge_color='deepskyblue', node_color='cyan', linewidth=1.0, node_size=1.5,
    title=f'Oracle Selection (Greedy, {len(selected_greedy_keys)} edges)\n'
          f'Avg={oracle_avg:.2f} px  Pred→GT={oracle_p2g:.2f}  GT→Pred={oracle_g2p:.2f}'
)
draw_graph_on_ax(
    axes[1, 1], green, all_edges_graph,
    edge_color='gold', node_color='cyan', linewidth=0.6, node_size=1.0,
    title=f'All Edges (r={SEARCH_RADIUS:.0f})  Hausdorff={all_edges_dist:.2f} px'
)

fig.suptitle(f'Oracle Connection Selection — {IMAGE_ID}', fontsize=14)
plt.show()

In [ ]:
# Per-point distance distributions for baseline vs oracle (edge points only)
oracle_edge_pts_dist = extract_edge_points(oracle_graph)
min_b_p2g, min_b_g2p = compute_point_min_distances(baseline_pts, gt_points)
min_o_p2g, min_o_g2p = compute_point_min_distances(oracle_edge_pts_dist, gt_points)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

BINS = 60
MAX_DIST = 50

for ax, b_dists, o_dists, direction in [
    (axes[0], min_b_p2g, min_o_p2g, 'Pred → GT'),
    (axes[1], min_b_g2p, min_o_g2p, 'GT → Pred'),
]:
    ax.hist(np.clip(b_dists, 0, MAX_DIST), bins=BINS, alpha=0.5, color='steelblue',
            label=f'Baseline  mean={b_dists.mean():.2f} px', density=True)
    ax.hist(np.clip(o_dists, 0, MAX_DIST), bins=BINS, alpha=0.5, color='limegreen',
            label=f'Oracle  mean={o_dists.mean():.2f} px', density=True)
    ax.set_xlabel('Min Distance to Counterpart (px)')
    ax.set_ylabel('Density')
    ax.set_title(f'Distance Distribution: {direction}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'Per-Point Distance Distributions: Baseline vs Oracle (Greedy, {len(selected_greedy_keys)} edges)'
    f'  — edge points only', fontsize=12)
plt.show()

In [ ]:
# Final overlay: image + annotation (transparent) + Oracle graph on top
fig, ax = plt.subplots(figsize=(72, 48), constrained_layout=True)

ax.imshow(green, cmap='gray')
annotation_overlay = np.ma.masked_where(annotation == 0, annotation)
ax.imshow(annotation_overlay, cmap='Reds', alpha=0.45)

for u, v, data in oracle_graph.edges(data=True):
    path = data.get('path', [u, v])
    if len(path) >= 2:
        ax.plot([p[1] for p in path], [p[0] for p in path],
                color='deepskyblue', linewidth=0.8, alpha=0.9)

ax.set_title(
    f'Oracle Graph on Annotation Overlay — {IMAGE_ID}\n'
    f'gray: image  |  red: annotation (α=0.45)  |  blue: oracle edges '
    f'(n={oracle_graph.number_of_edges()}, greedy)\n'
    f'Avg Hausdorff = {oracle_avg:.2f} px  '
    f'(Pred→GT={oracle_p2g:.2f}, GT→Pred={oracle_g2p:.2f})',
    fontsize=11
)
ax.axis('off')
plt.show()